# 实验二：房源采集与 MySQL 存储

教学站：<http://www.bspider.top/anjuke/>

采集 8 个字段：`title / house / address / struct / area / floor / maketime / price`

环境依赖已具备（requests 2.34.2、beautifulsoup4 4.15.0、PyMySQL 2.2.8、lxml），无需额外安装。
如确需安装，在单元中运行：

```python
%pip install requests beautifulsoup4 pymysql
```

## 第一阶段：数据库连接自检

确认 MySQL 服务可用、库为 `smartpark_lab`、`houseinfo` 表已建立。

In [1]:
import pymysql

with pymysql.connect(
    host="mysql", port=3306,
    user="student", password="123456",
    database="smartpark_lab", charset="utf8mb4", connect_timeout=5
) as conn:
    with conn.cursor() as cursor:
        cursor.execute("SELECT VERSION(), DATABASE(), CURRENT_USER()")
        print(cursor.fetchone())

        cursor.execute("SHOW TABLES")
        print("tables:", cursor.fetchall())

        cursor.execute("DESCRIBE houseinfo")
        for row in cursor.fetchall():
            print(row)

        cursor.execute("SELECT COUNT(*) AS before_count FROM houseinfo")
        print("before_count =", cursor.fetchone()[0])

('8.4.11', 'smartpark_lab', 'student@%')
tables: (('houseinfo',),)
('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('title', 'varchar(200)', 'YES', '', None, '')
('house', 'varchar(50)', 'YES', '', None, '')
('address', 'varchar(200)', 'YES', '', None, '')
('struct', 'varchar(50)', 'YES', '', None, '')
('area', 'varchar(20)', 'YES', '', None, '')
('floor', 'varchar(20)', 'YES', '', None, '')
('maketime', 'varchar(20)', 'YES', '', None, '')
('price', 'varchar(20)', 'YES', '', None, '')
before_count = 0


## 第二阶段：房源采集与 CSV 数据存储

### 1. 获取第一页

教学站有反爬校验：请求头不完整时会返回「访问验证」页（状态码仍为 200，但 `div.property` 为 0）。
因此下面使用**本机 Chrome 155 的完整请求头**。

In [26]:
import time
from pathlib import Path
from urllib.parse import urljoin

import csv
import requests
from bs4 import BeautifulSoup

# 本机 Chrome 版本 155.0.8059.85（Windows 11），使用真实浏览器请求头
UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/155.0.0.0 Safari/537.36")

HEADERS = {
    "UA":"User-Agent",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,"
              "image/avif,image/webp,image/apng,*/*;q=0.8,"
              "application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
    "Accept-Encoding": "gzip, deflate",
    "Referer": "http://www.bspider.top/",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Sec-Fetch-User": "?1",
    "sec-ch-ua": '"Chromium";v="155", "Google Chrome";v="155", "Not_A Brand";v="24"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "Connection": "keep-alive",
}
print(UA)

Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/155.0.0.0 Safari/537.36


In [3]:
# 请求第一页并检查返回内容
BASE = "http://www.bspider.top/anjuke/"
r = requests.get(BASE, headers=HEADERS, timeout=30)
r.raise_for_status()
r.encoding = r.apparent_encoding
soup = BeautifulSoup(r.text, "html.parser")

print("status =", r.status_code)
print("title  =", soup.title.get_text(strip=True) if soup.title else None)
rows = soup.select("div.property")
print("div.property =", len(rows))

assert "访问验证" not in (soup.title.get_text() if soup.title else ""), "命中访问验证页，停止采集"
assert len(rows) > 0, "房源列表为空，停止采集"

status = 200
title  = 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
div.property = 60


**自己检查：** 状态码 200、标题为「重庆二手房…」、`div.property` 为 60。
若出现「访问验证」或 0 条，说明被拦截，记录错误并停止，不要改采真实安居客网站。

### 2. 分析网页，提取 8 个字段

先打印第一条房源的属性组，**确认每项顺序**——尤其不要把「朝向」当成「楼层」。

In [4]:
# 确认属性组顺序
for i, p in enumerate(rows[0].select("p.property-content-info-text")):
    print(i, repr(p.get_text(strip=True)))

0 '3室2厅1卫'
1 '89.9㎡'
2 '南北'
3 '低层(共27层)'
4 '2019年建造'


In [15]:
FIELDS = ["title", "house", "address", "struct", "area",
          "floor", "maketime", "price"]


def text_of(node):
    """取节点文字；节点不存在时返回空字符串。"""
    return node.get_text(strip=True) if node else ""


def parse_row(row):
    """解析一条房源，按 FIELDS 顺序返回 8 个值。"""
    title = text_of(row.select_one("h3.property-content-title-name"))
    house = text_of(row.select_one("p.property-content-info-comm-name"))
    # 地址分散在多个 span 中，直接取完整文字，不额外插入空格
    address = text_of(row.select_one("p.property-content-info-comm-address"))

    # 属性组按 [户型, 面积, 朝向, 楼层, 年代] 顺序出现
    texts = [p.get_text(strip=True)
             for p in row.select("p.property-content-info-text")]
    struct = texts[0] if len(texts) > 0 else ""
    area = texts[1] if len(texts) > 1 else ""
    # 下标 2 是朝向（如“南北”），不能当成楼层
    floor = texts[3] if len(texts) > 3 else ""
    maketime = texts[4] if len(texts) > 4 else ""   # 缺失时为空字符串

    # 单价（元/㎡），不是 property-price-total 的总价万元
    price = text_of(row.select_one("p.property-price-average"))
    return [title, house, address, struct, area, floor, maketime, price]


def show(rec, tag=""):
    print(f"--- {tag} ---")
    for k, v in zip(FIELDS, rec):
        print(f"  {k:9s} = {v}")


show(parse_row(rows[0]), "第 1 条")

--- 第 1 条 ---
  title     = 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
  house     = 万科金域学府翰林菁华里
  address   = 南岸学府大道二塘路59号
  struct    = 3室2厅1卫
  area      = 89.9㎡
  floor     = 低层(共27层)
  maketime  = 2019年建造
  price     = 16465元/㎡


In [17]:
# 对照网页检查第 2 条和最后一条
show(parse_row(rows[1]), "第 2 条")
show(parse_row(rows[-1]), f"最后一条（第 {len(rows)} 条）")

--- 第 2 条 ---
  title     = 黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住
  house     = 两代一家
  address   = 渝北渝北黄泥磅紫荆路29号
  struct    = 3室2厅2卫
  area      = 94.5㎡
  floor     = 低层(共28层)
  maketime  = 2008年建造
  price     = 14815元/㎡
--- 最后一条（第 60 条） ---
  title     = 精装修 电梯房 4室2厅 91平
  house     = 华润凯旋天地
  address   = 南岸弹子石腾龙大道45号
  struct    = 4室2厅2卫
  area      = 91.3㎡
  floor     = 低层(共32层)
  maketime  = 2015年建造
  price     = 15886元/㎡


In [18]:
# 本页建造年代缺失情况
missing = sum(1 for row in rows if parse_row(row)[6] == "")
print("本页条数:", len(rows), " maketime 缺失:", missing)

本页条数: 60  maketime 缺失: 0


**截图：** 框出 `select()` / `select_one()`，以及面积、单价、年代的定位代码。

### 3. 采集下一页

下一页按钮：`a.next`；可点击时带 `next-active` class，末页只剩 `a.next` 且 `href="#"`。

In [19]:
# 查看下一页链接，并用 urljoin 补全
next_a = soup.select_one("a.next")
print("a.next:", next_a)
href = next_a.get("href")
print("href 原值:", href)
print("class   :", next_a.get("class"))
print("urljoin 后:", urljoin(BASE, href))

a.next: <a class="next next-active" data-npv="" data-v-29d65fc6="" href="?page=2">下一页</a>
href 原值: ?page=2
class   : ['next', 'next-active']
urljoin 后: http://www.bspider.top/anjuke/?page=2


In [20]:
def fetch_page(url):
    """请求一页，返回 soup；失败时抛异常。"""
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding
    return BeautifulSoup(resp.text, "html.parser")


def collect(max_pages=20, sleep=1.0):
    """从首页开始翻页采集，返回结果列表和停止原因。"""
    out_list = []                      # 结果列表放在函数内，不在全局
    url, seen, reason = BASE, set(), ""

    while url and len(seen) < max_pages:
        if url in seen:
            reason = f"重复地址，中止：{url}"
            break
        seen.add(url)

        page = fetch_page(url)
        rows = page.select("div.property")
        if not rows:
            reason = f"空列表，中止：{url}"
            break
        for row in rows:
            out_list.append(parse_row(row))
        print(f"page={len(seen)} url={url} rows={len(rows)} cum={len(out_list)}")

        # 分页区域消失 = 结构异常，不算采集完成
        next_a = page.select_one("a.next")
        if next_a is None:
            reason = "分页区域缺失，结构异常，中止"
            break
        href = next_a.get("href")
        cls = next_a.get("class", [])
        # 末页：按钮仍在但没有 next-active，且 href 为 #
        if "next-active" not in cls or href in (None, "", "#"):
            reason = "到达末页（下一页按钮已禁用）"
            break

        url = urljoin(url, href)
        if not url.startswith(BASE):
            reason = f"链接越出教学站，中止：{url}"
            break
        time.sleep(sleep)
    else:
        reason = f"达到安全页数上限 {max_pages}，记为中止，不算采集完成"

    print("停止原因:", reason)
    return out_list, reason


# 先试采 2 页，确认第二页没有重复第一页
test_list, test_reason = collect(max_pages=2)
print("试采条数:", len(test_list))
print("前两条:", test_list[0][:2])
print("第61条:", test_list[60][:2])
print("两页是否有重复:", len(test_list) != len({tuple(x) for x in test_list}))

page=1 url=http://www.bspider.top/anjuke/ rows=60 cum=60
page=2 url=http://www.bspider.top/anjuke/?page=2 rows=60 cum=120
停止原因: 达到安全页数上限 2，记为中止，不算采集完成
试采条数: 120
前两条: ['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里']
第61条: ['万科物业自营 朝中庭 业主诚心出售', '万科金域学府翰林菁华里']
两页是否有重复: True


In [21]:
# 确认无误后全量采集
records, stop_reason = collect(max_pages=20)
print("采集记录数（去重前）:", len(records))
print("停止原因:", stop_reason)

page=1 url=http://www.bspider.top/anjuke/ rows=60 cum=60
page=2 url=http://www.bspider.top/anjuke/?page=2 rows=60 cum=120
page=3 url=http://www.bspider.top/anjuke/?page=3 rows=60 cum=180
page=4 url=http://www.bspider.top/anjuke/?page=4 rows=60 cum=240
page=5 url=http://www.bspider.top/anjuke/?page=5 rows=60 cum=300
page=6 url=http://www.bspider.top/anjuke/?page=6 rows=60 cum=360
page=7 url=http://www.bspider.top/anjuke/?page=7 rows=60 cum=420
page=8 url=http://www.bspider.top/anjuke/?page=8 rows=60 cum=480
page=9 url=http://www.bspider.top/anjuke/?page=9 rows=60 cum=540
page=10 url=http://www.bspider.top/anjuke/?page=10 rows=60 cum=600
停止原因: 到达末页（下一页按钮已禁用）
采集记录数（去重前）: 600
停止原因: 到达末页（下一页按钮已禁用）


**截图：** 框出 `href`、`urljoin`、循环更新地址和停止条件，并附翻页运行日志。

### 4. 保存和核对 CSV

In [22]:
# 保存 CSV：utf-8-sig + newline=""
Path("data").mkdir(exist_ok=True)
CSV_PATH = Path("data") / "houseinfo.csv"

with open(CSV_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)
    writer.writerows(records)          # 不查重，保存全部采集记录
print("已保存:", CSV_PATH)

已保存: data/houseinfo.csv


In [23]:
# 回读核对
with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    rows_back = list(csv.DictReader(f))
print("回读数据行数（不含表头）:", len(rows_back))
print("表头:", list(rows_back[0].keys()))

show([rows_back[0][k] for k in FIELDS], "CSV 第 1 条")
show([rows_back[60][k] for k in FIELDS], "CSV 跨页第 61 条")
show([rows_back[-1][k] for k in FIELDS], "CSV 最后一条")

回读数据行数（不含表头）: 600
表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
--- CSV 第 1 条 ---
  title     = 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
  house     = 万科金域学府翰林菁华里
  address   = 南岸学府大道二塘路59号
  struct    = 3室2厅1卫
  area      = 89.9㎡
  floor     = 低层(共27层)
  maketime  = 2019年建造
  price     = 16465元/㎡
--- CSV 跨页第 61 条 ---
  title     = 万科物业自营 朝中庭 业主诚心出售
  house     = 万科金域学府翰林菁华里
  address   = 南岸学府大道二塘路59号
  struct    = 3室2厅1卫
  area      = 85.7㎡
  floor     = 低层(共27层)
  maketime  = 2018年建造
  price     = 16914元/㎡
--- CSV 最后一条 ---
  title     = 恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售
  house     = 恒大照母山山水城
  address   = 渝北照母山恒康路9号
  struct    = 3室2厅1卫
  area      = 90.2㎡
  floor     = 低层(共32层)
  maketime  = 2015年建造
  price     = 15530元/㎡


**截图：** 框出 CSV 保存与回读代码、回读条数、样本核对结果。

记下「去重前 / 去重后 / CSV 数据行数」，填进 REPORT.md 的核对表。

## 第三阶段：MySQL 数据存储

（待填写：读取 `data/houseinfo.csv` 写入 `smartpark_lab.houseinfo`）

In [1]:
import csv
from pathlib import Path

FIELDS = ["title", "house", "address", "struct", "area",
          "floor", "maketime", "price"]

CSV_PATH = Path("data") / "houseinfo.csv"
with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames                 # 检查表头
    print("表头:", header)
    assert header == FIELDS, "表头与8个字段不一致"
    records = [[row[k] for k in FIELDS] for row in reader]

print("CSV 数据行数:", len(records))
print("每条字段数:", set(len(r) for r in records))   # 应为 {8}
print("第一条:", records[0])


表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
CSV 数据行数: 600
每条字段数: {8}
第一条: ['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡']


In [4]:
import pymysql

INSERT_SQL = """
INSERT INTO houseinfo
    (title, house, address, struct, area, floor, maketime, price)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""


def insert_records(records):
    """把 records 写入数据库，返回本次实际新增条数。重复运行不重复插入。"""
    conn = cursor = None
    try:
        conn = pymysql.connect(
            host="mysql", port=3306,
            database="smartpark_lab",
            user="student", password="123456",
            charset="utf8mb4", autocommit=False,   # 由程序提交事务
            connect_timeout=5,
        )
        cursor = conn.cursor()

        # 防重复：先读库内已有记录，按8个字段比较，只插尚不存在的
        cursor.execute(
            "SELECT title, house, address, struct, area, "
            "floor, maketime, price FROM houseinfo"
        )
        existing = {tuple(row) for row in cursor.fetchall()}
        pending = [r for r in records if tuple(r) not in existing]
        print("库内已有:", len(existing), " 本批待插入:", len(pending))

        if pending:
            affected = cursor.executemany(INSERT_SQL, pending)
            print("executemany 受影响行数:", affected)
            conn.commit()                          # 成功后提交
        return len(pending)

    except Exception as e:
        if conn is not None:
            try:
                conn.rollback()                    # 出错时回滚
            except Exception:
                pass
        print("入库失败，已回滚:", e)
        raise
    finally:
        if cursor is not None:
            cursor.close()                         # 先关游标
        if conn is not None:
            conn.close()                           # 再关连接


In [5]:
added = insert_records(records[:3])
print("少量测试新增:", added)


库内已有: 0  本批待插入: 3
executemany 受影响行数: 3
少量测试新增: 3


In [6]:
added = insert_records(records)
print("全量入库新增:", added)


库内已有: 3  本批待插入: 597
executemany 受影响行数: 597
全量入库新增: 597


In [7]:
added_again = insert_records(records)
print("重跑新增:", added_again)     # 应为 0


库内已有: 498  本批待插入: 0
重跑新增: 0


In [9]:
conn = pymysql.connect(
    host="mysql", port=3306, database="smartpark_lab",
    user="student", password="123456",
    charset="utf8mb4", autocommit=False, connect_timeout=5,
)
cursor = conn.cursor()
cursor.execute(
    "SELECT title, house, address, struct, area, floor, "
    "maketime, price FROM houseinfo ORDER BY id"
)
rows = cursor.fetchall()
print("库内总条数:", len(rows))
for i, tag in ((0, "第1条"), (60, "第61条(跨页)"), (-1, "最后一条")):
    print(f"--- {tag} ---")
    for k, v in zip(FIELDS, rows[i]):
        print(f"  {k:9s} = {v}")
    print("  与CSV一致:", list(rows[i]) == records[i])
cursor.close()
conn.close()


库内总条数: 600
--- 第1条 ---
  title     = 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
  house     = 万科金域学府翰林菁华里
  address   = 南岸学府大道二塘路59号
  struct    = 3室2厅1卫
  area      = 89.9㎡
  floor     = 低层(共27层)
  maketime  = 2019年建造
  price     = 16465元/㎡
  与CSV一致: True
--- 第61条(跨页) ---
  title     = 万科物业自营 朝中庭 业主诚心出售
  house     = 万科金域学府翰林菁华里
  address   = 南岸学府大道二塘路59号
  struct    = 3室2厅1卫
  area      = 85.7㎡
  floor     = 低层(共27层)
  maketime  = 2018年建造
  price     = 16914元/㎡
  与CSV一致: True
--- 最后一条 ---
  title     = 恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售
  house     = 恒大照母山山水城
  address   = 渝北照母山恒康路9号
  struct    = 3室2厅1卫
  area      = 90.2㎡
  floor     = 低层(共32层)
  maketime  = 2015年建造
  price     = 15530元/㎡
  与CSV一致: True
